# 03 · Gold: modelo para o Power BI

Star schema pronto para o modelo semântico em **Direct Lake**:

| Tabela | Grão | Para quê |
|---|---|---|
| `fato_tarefa` | uma linha por tarefa | volume, prazo, esforço (estimado × apontado), lead time e ciclo |
| `fato_passagem_status` | uma linha por passagem da tarefa por uma etapa | tempo em cada etapa, gargalos, bloqueios, fluxo do mês |
| `dim_projeto` | um projeto | status, equipe, gestor, prazo planejado × real |
| `dim_pessoa` | uma pessoa | responsável pelas tarefas |
| `dim_status` | uma etapa | ordem e categoria das etapas |
| `dim_data` | um dia | calendário em português |
| `ref_parametros` | uma linha | data de referência dos dados |

As métricas de tempo saem do **histórico de status**, não de campos prontos da API: ciclo, dias bloqueada e retrabalho
são recalculados a partir dos eventos, então batem entre si e com o fluxo mostrado no painel.

Datas e horas ficam no **horário de Brasília** e tudo é medido contra a **data de referência** gravada nos dados,
não contra o relógio: o painel não muda sozinho de um dia para o outro.

In [ ]:
FORMATO_TABELA = "delta"   # delta no Fabric; o teste local usa parquet

In [ ]:
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
FUSO = "America/Sao_Paulo"
FINAIS = ["Concluída", "Cancelada"]


def local(coluna: str):
    """Instante UTC -> data e hora de Brasília (é o que o Power BI vai mostrar)."""
    return F.from_utc_timestamp(F.col(coluna), FUSO)


def dias_entre(inicio, fim):
    return F.round((F.unix_timestamp(fim) - F.unix_timestamp(inicio)) / 86400, 2)


ref = spark.table("silver_meta").first()
DATA_REF = ref["data_referencia"]
FIM_REF = F.to_timestamp(F.lit(f"{DATA_REF} 23:59:59"))   # fim do dia de referência, já em horário local
print(f"data de referência: {DATA_REF}")

equipes = spark.table("silver_equipes")
pessoas = spark.table("silver_pessoas")
projetos = spark.table("silver_projetos")
tarefas = spark.table("silver_tarefas")
historico = spark.table("silver_historico")

In [ ]:
# fato_passagem_status: cada evento abre uma passagem que termina no evento seguinte da mesma tarefa
ordem = Window.partitionBy("tarefa_id").orderBy("ocorrido_em", "id")
passagens = (historico
    .withColumn("entrada_em", local("ocorrido_em"))
    .withColumn("saida_em", F.lead("entrada_em").over(ordem))
    .withColumn("sequencia", F.row_number().over(ordem))
    .join(tarefas.select(F.col("id").alias("tarefa_id"), "projeto_id"), "tarefa_id")
    .withColumn("etapa_final", F.col("status_novo").isin(FINAIS))
    .withColumn("etapa_atual", F.col("saida_em").isNull())
    # etapa final não tem duração; etapa ainda aberta conta até a data de referência
    .withColumn("dias_na_etapa", F.when(F.col("etapa_final"), F.lit(None).cast("double"))
                .otherwise(dias_entre(F.col("entrada_em"), F.coalesce("saida_em", FIM_REF))))
    .select(F.col("id").alias("passagem_id"), "tarefa_id", "projeto_id", "sequencia",
            F.col("status_anterior"), F.col("status_novo").alias("status"), "pessoa_id",
            "entrada_em", "saida_em", F.to_date("entrada_em").alias("data_entrada"),
            "dias_na_etapa", "etapa_atual", "etapa_final"))

# indicadores por tarefa a partir das passagens
por_tarefa = passagens.groupBy("tarefa_id").agg(
    F.min(F.when(F.col("status") == "Em Andamento", F.col("entrada_em"))).alias("inicio_execucao_em"),
    F.sum(F.when(F.col("status") == "Bloqueada", 1).otherwise(0)).alias("qtd_bloqueios"),
    F.round(F.sum(F.when(F.col("status") == "Bloqueada", F.col("dias_na_etapa")).otherwise(0)), 2)
        .alias("dias_bloqueada"),
    F.sum(F.when((F.col("status_anterior") == "Em Revisão") & (F.col("status") == "Em Andamento"), 1)
          .otherwise(0)).alias("qtd_retrabalho"),
    F.max("entrada_em").alias("ultima_mudanca_em"),
)

In [ ]:
t = (tarefas
     .withColumn("criada_em", local("criada_em"))
     .withColumn("concluida_em", local("concluida_em"))
     .join(por_tarefa, tarefas["id"] == por_tarefa["tarefa_id"], "left"))

aberta = ~F.col("status").isin(FINAIS)
concluida = F.col("status") == "Concluída"
data_conclusao = F.to_date("concluida_em")

fato_tarefa = t.select(
    F.col("id").alias("tarefa_id"), "projeto_id", "responsavel_id", "titulo", "tipo", "prioridade", "status",
    F.to_date("criada_em").alias("data_criacao"), "criada_em",
    F.to_date("inicio_execucao_em").alias("data_inicio_execucao"),
    data_conclusao.alias("data_conclusao"), "concluida_em",
    F.col("prazo").alias("data_prazo"),
    F.col("estimativa_horas"), F.col("horas_apontadas"),
    F.when(concluida, F.round(F.col("horas_apontadas") - F.col("estimativa_horas"), 1)).alias("desvio_horas"),
    aberta.alias("aberta"),
    F.when(concluida, dias_entre(F.col("criada_em"), F.col("concluida_em"))).alias("lead_time_dias"),
    F.when(concluida, dias_entre(F.col("inicio_execucao_em"), F.col("concluida_em"))).alias("ciclo_dias"),
    F.when(aberta, dias_entre(F.col("criada_em"), FIM_REF)).alias("idade_dias"),
    F.when(aberta, dias_entre(F.col("ultima_mudanca_em"), FIM_REF)).alias("dias_na_etapa_atual"),
    F.coalesce("qtd_bloqueios", F.lit(0)).alias("qtd_bloqueios"),
    F.coalesce("dias_bloqueada", F.lit(0.0)).alias("dias_bloqueada"),
    F.coalesce("qtd_retrabalho", F.lit(0)).alias("qtd_retrabalho"),
    F.when(F.col("status") == "Cancelada", "Cancelada")
     .when(F.col("prazo").isNull(), "Sem prazo")
     .when(concluida & (data_conclusao <= F.col("prazo")), "Concluída no prazo")
     .when(concluida, "Concluída com atraso")
     .when(F.col("prazo") < F.lit(DATA_REF), "Vencida")
     .otherwise("No prazo").alias("situacao_prazo"),
    F.when(concluida, F.greatest(F.datediff(data_conclusao, "prazo"), F.lit(0)))
     .when(aberta, F.greatest(F.datediff(F.lit(DATA_REF), "prazo"), F.lit(0)))
     .alias("dias_atraso"),
)

In [ ]:
gestores = pessoas.select(F.col("id").alias("gestor_id"), F.col("nome").alias("gestor"))
nomes_equipe = equipes.select(F.col("id").alias("equipe_id"), F.col("nome").alias("equipe"))
ativo_ou_planejado = F.col("status").isin("Em Andamento", "Planejamento")
fim_real_ou_ref = F.coalesce("data_conclusao", F.lit(DATA_REF).cast("date"))

dim_projeto = (projetos
    .join(nomes_equipe, "equipe_id", "left")
    .join(gestores, "gestor_id", "left")
    .select(
        F.col("id").alias("projeto_id"), F.col("nome").alias("projeto"), "equipe", "gestor", "prioridade",
        "status", "data_inicio", "data_fim_planejada", "data_conclusao", "horas_orcadas",
        F.datediff("data_fim_planejada", "data_inicio").alias("dias_planejados"),
        F.when(F.col("status") == "Concluído",
               F.when(F.col("data_conclusao") <= F.col("data_fim_planejada"), "Concluído no prazo")
                .otherwise("Concluído com atraso"))
         .when(ativo_ou_planejado & (F.lit(DATA_REF) > F.col("data_fim_planejada")), "Atrasado")
         .when(ativo_ou_planejado, "No prazo")
         .otherwise(F.col("status")).alias("situacao_prazo"),
        F.when(F.col("status").isin("Concluído", "Em Andamento", "Planejamento"),
               F.greatest(F.datediff(fim_real_ou_ref, "data_fim_planejada"), F.lit(0))).alias("dias_atraso"),
        # quanto do prazo planejado já passou (só para projetos vivos)
        F.when(ativo_ou_planejado,
               F.round(F.datediff(F.lit(DATA_REF), "data_inicio") / F.datediff("data_fim_planejada", "data_inicio"), 4))
         .alias("percentual_prazo_decorrido"),
    ))

dim_pessoa = (pessoas.join(nomes_equipe, "equipe_id", "left")
    .select(F.col("id").alias("pessoa_id"), F.col("nome").alias("pessoa"), "equipe", "cargo", "ativo",
            "data_admissao"))

dim_status = spark.createDataFrame(
    [("Backlog", "Não iniciada", 1), ("A Fazer", "Não iniciada", 2), ("Em Andamento", "Em execução", 3),
     ("Bloqueada", "Em execução", 4), ("Em Revisão", "Em execução", 5), ("Concluída", "Concluída", 6),
     ("Cancelada", "Cancelada", 7)],
    "status string, categoria string, ordem int")

In [ ]:
limites = (fato_tarefa.select(F.min("data_criacao").alias("d")).union(dim_projeto.select(F.min("data_inicio")))
           .union(dim_projeto.select(F.max("data_fim_planejada"))).union(spark.createDataFrame([(DATA_REF,)], "d date")))
inicio_cal, fim_cal = limites.agg(F.min("d"), F.max("d")).first()
inicio_cal, fim_cal = inicio_cal.replace(month=1, day=1), fim_cal.replace(month=12, day=31)   # anos completos

MESES = F.array(*[F.lit(m) for m in ("jan", "fev", "mar", "abr", "mai", "jun",
                                      "jul", "ago", "set", "out", "nov", "dez")])
DIAS = F.array(*[F.lit(d) for d in ("seg", "ter", "qua", "qui", "sex", "sáb", "dom")])
dia_semana = (F.dayofweek("data") + 5) % 7 + 1   # 1 = segunda ... 7 = domingo

dim_data = (spark.sql(f"SELECT explode(sequence(DATE'{inicio_cal}', DATE'{fim_cal}')) AS data")
    .select(
        "data", F.year("data").alias("ano"), F.month("data").alias("mes"),
        F.element_at(MESES, F.month("data")).alias("mes_nome"),
        F.concat(F.element_at(MESES, F.month("data")), F.lit("/"), F.date_format("data", "yy")).alias("mes_ano"),
        (F.year("data") * 100 + F.month("data")).alias("ano_mes"),   # ordenação de mes_ano
        F.concat(F.lit("T"), F.quarter("data")).alias("trimestre"),
        F.date_sub("data", dia_semana - 1).alias("semana_inicio"),
        dia_semana.alias("dia_semana"), F.element_at(DIAS, dia_semana).alias("dia_semana_nome"),
        (dia_semana <= 5).alias("dia_util"),
        (F.col("data") <= F.lit(DATA_REF)).alias("ate_referencia"),
    ))

ref_parametros = (spark.createDataFrame([(DATA_REF,)], "data_referencia date")
                  .withColumn("processado_em", F.from_utc_timestamp(F.current_timestamp(), FUSO)))

In [ ]:
def salvar(df: DataFrame, nome: str) -> int:
    df.write.format(FORMATO_TABELA).mode("overwrite").option("overwriteSchema", "true").saveAsTable(nome)
    return spark.table(nome).count()


saida = {"fato_tarefa": fato_tarefa, "fato_passagem_status": passagens, "dim_projeto": dim_projeto,
         "dim_pessoa": dim_pessoa, "dim_status": dim_status, "dim_data": dim_data,
         "ref_parametros": ref_parametros}
for nome, df in saida.items():
    print(f"{nome:<22} {salvar(df, nome):>7}")

In [ ]:
# Conferências: o modelo só é publicado se as contas fecharem
checagens = {
    "toda tarefa tem projeto": spark.sql("""SELECT count(*) FROM fato_tarefa f
        LEFT ANTI JOIN dim_projeto p ON f.projeto_id = p.projeto_id""").first()[0] == 0,
    "todo responsável existe": spark.sql("""SELECT count(*) FROM fato_tarefa f
        LEFT ANTI JOIN dim_pessoa p ON f.responsavel_id = p.pessoa_id""").first()[0] == 0,
    "status da tarefa = última passagem": spark.sql("""SELECT count(*) FROM fato_tarefa f
        JOIN fato_passagem_status p ON p.tarefa_id = f.tarefa_id AND p.etapa_atual
        WHERE p.status <> f.status""").first()[0] == 0,
    "uma etapa atual por tarefa": spark.sql("""SELECT count(*) FROM (SELECT tarefa_id FROM fato_passagem_status
        WHERE etapa_atual GROUP BY tarefa_id HAVING count(*) <> 1)""").first()[0] == 0,
    "tempos não negativos": spark.sql("""SELECT count(*) FROM fato_tarefa
        WHERE ciclo_dias < 0 OR lead_time_dias < 0 OR dias_bloqueada < 0""").first()[0] == 0,
    "ciclo ≤ lead time": spark.sql("""SELECT count(*) FROM fato_tarefa
        WHERE ciclo_dias > lead_time_dias""").first()[0] == 0,
    "calendário cobre todas as datas": spark.sql("""SELECT count(*) FROM fato_tarefa f
        LEFT ANTI JOIN dim_data d ON f.data_criacao = d.data""").first()[0] == 0,
}
for nome, ok in checagens.items():
    print(("ok    " if ok else "FALHOU") + f"  {nome}")
falhas = [n for n, ok in checagens.items() if not ok]
if falhas:
    raise AssertionError(f"gold inconsistente: {falhas}")